# Аптамерный докинг: RhoFold+ → RDKit/Meeko → Vina

**Runtime:** GPU (T4) рекомендуется для RhoFold+.

**Порядок:** ячейки сверху вниз → *Runtime → Run all*.

| Шаг | Что делает |
|-----|------------|
| 0 | Установка системных пакетов |
| 1 | Conda: RhoFold+, dockprep, Vina |
| 2 | Параметры (sequence + SMILES) |
| 3 | RhoFold+ → 3D PDB |
| 4 | Подготовка рецептора |
| 5 | Лиганд SMILES → PDBQT (RDKit + Meeko) |
| 6 | Multi-site AutoDock Vina |
| 7 | 3D-визуализация |


In [1]:
# === 0. Системные пакеты ===
!apt-get -qq update
!apt-get -qq install -y openbabel
!pip -q install py3Dmol numpy pandas

import os
from pathlib import Path

# Miniconda (если ещё нет)
if not Path("/content/miniconda3/bin/conda").exists():
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p /content/miniconda3

os.environ["PATH"] = "/content/miniconda3/bin:" + os.environ.get("PATH", "")
print("System packages: OK")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.2.4-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.2.4-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-6ubuntu5) ...
Selecting previousl

In [2]:
# === 1a. Conda ToS + RhoFold+ ===
import subprocess
from pathlib import Path

CONDA = "/content/miniconda3/bin/conda"
RHOFOLD_DIR = Path("/content/RhoFold")
ENV_RHOFOLD = "rhofold"

def env_exists(name):
    r = subprocess.run([CONDA, "env", "list"], capture_output=True, text=True)
    return any(line.split()[0] == name for line in r.stdout.splitlines() if line.strip())

for ch in [
    "https://repo.anaconda.com/pkgs/main",
    "https://repo.anaconda.com/pkgs/r",
]:
    subprocess.run([CONDA, "tos", "accept", "--override-channels", "--channel", ch], check=False)

if not RHOFOLD_DIR.exists():
    !git clone -q https://github.com/ml4bio/RhoFold.git {RHOFOLD_DIR}

if not env_exists(ENV_RHOFOLD):
    print("Creating rhofold env (~15-30 min)...")
    !{CONDA} env create -f {RHOFOLD_DIR}/envs/environment_linux.yaml
    !cd {RHOFOLD_DIR} && {CONDA} run -n {ENV_RHOFOLD} python setup.py install

if not env_exists(ENV_RHOFOLD):
    raise RuntimeError("Conda env 'rhofold' not created — проверьте ошибки выше")

CKPT = RHOFOLD_DIR / "pretrained" / "rhofold_pretrained_params.pt"
CKPT.parent.mkdir(parents=True, exist_ok=True)
if not CKPT.exists():
    !wget -q https://huggingface.co/cuhkaih/rhofold/resolve/main/rhofold_pretrained_params.pt -O {CKPT}

print("RhoFold+ env: OK")
print("Checkpoint:", CKPT.exists())


Creating rhofold env (~15-30 min)...
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / done
Channels:
 - pytorch
 - defaults
 - conda-forge
Platform: linux-64
Solving environment: | / done

pytorch-1.10.2       | 1.21 GB   | :   0% 0/1 [00:00<?, ?it/s]
cudatoolkit-11.3.1   | 816.4 MB  | :   0% 0/1 [00:00<?, ?it/s]

mkl-2022.1.0         | 199.6 MB  | :   0% 0/1 [00:00<?, ?it/s]


python-3.7.12        | 57.3 MB   | :   0% 0/1 [00:00<?, ?it/s]



libboost-1.73.0      | 13.8 MB   | :   0% 0/1 [00:00<?, ?it/s]




openmm-7.7.0         | 11.3 MB   | :   0% 0/1 [00:00<?, ?it/s]





arrow-cpp-8.0.0      | 10.6 MB   | :   0% 0/1 [00:00<?, ?it/s]






icu-58.2             | 10.5 MB   | :   0% 0/1 [00:00<?, ?it/s]







pandas-1.3.5         | 9.3 MB    | :   0% 0/1 [00:00<?, ?it/s]








numpy-1.21.6         | 6.1 MB    | :   0% 0/1 [00:00<?, ?it/s]









llvm-openmp-14.0.4   | 5.8 MB    | :   0% 0/1 [00:00<?, ?

In [3]:
# === 1b. Conda: dockprep (RDKit + Meeko) + docking (Vina CLI) ===
ENV_DOCKPREP = "dockprep"
ENV_VINA = "docking"

if not env_exists(ENV_DOCKPREP):
    !{CONDA} create -y -n {ENV_DOCKPREP} python=3.11
    !{CONDA} install -y -n {ENV_DOCKPREP} -c conda-forge rdkit meeko numpy

if not env_exists(ENV_VINA):
    !{CONDA} create -y -n {ENV_VINA} python=3.11
    !{CONDA} install -y -n {ENV_VINA} -c conda-forge vina

DOCK_PYTHON = f"/content/miniconda3/envs/{ENV_DOCKPREP}/bin/python"
VINA_BIN = Path(f"/content/miniconda3/envs/{ENV_VINA}/bin/vina")

assert Path(DOCK_PYTHON).exists(), "dockprep python missing"
assert VINA_BIN.exists(), "vina binary missing"
print("dockprep:", DOCK_PYTHON)
print("vina CLI:", VINA_BIN)


Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: linux-64
Solving environment: \ done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /content/miniconda3/envs/dockprep

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-5.1          |           52_gnu           7 KB
    ca-certificates-2026.5.14  |       h06a4308_0         107 KB
    libexpat-2.8.1             |       h7354ed3_0         123 KB
    libgcc-15.2.0              |       h69a1729_8         803 KB
    libgcc-ng-15.2.0           |       h166f726_8          28 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    libstdcxx-15.2.0       

## 2. Параметры

Измените `RNA_SEQUENCE` и `LIGAND_SMILES` под свой эксперимент.

In [11]:
# === 2. CONFIG ===
import re
from pathlib import Path

WORKDIR = Path("/content/aptamer_docking")
WORKDIR.mkdir(parents=True, exist_ok=True)

# --- RNA aptamer ---
RNA_SEQUENCE = "AAAGCGUCGUCCCCACCUC "
JOB_NAME = "aptamer_rf"

# --- Ligand ---
LIGAND_SMILES = "Cn1c(=O)c2[nH]cnc2n(C)c1=O"
LIGAND_PH = 7.4

# --- RhoFold+ ---
RELAX_STEPS = 200
USE_SINGLE_SEQ = True  # без 900 GB MSA-баз

# --- Vina ---
LOCAL_BOX = [24, 24, 24]
BLIND_BOX = [40, 40, 40]
EXHAUSTIVENESS = 16
N_POSES = 5
CONTACT_CUTOFF = 4.0
MIN_CONTACTS = 15
CONSENSUS_CUTOFF = 10.0
ADD_RECEPTOR_H = False

# --- Paths ---
RHOFOLD_OUT = WORKDIR / "rhofold_out"
FASTA_PATH = WORKDIR / f"{JOB_NAME}.fasta"
APTAMER_PDB_RAW = RHOFOLD_OUT / "relaxed_200_model.pdb"  # обновится после inference
APTAMER_PDB_FIXED = WORKDIR / "aptamer_fixed.pdb"
APTAMER_PDB = WORKDIR / "aptamer_clean.pdb"
RECEPTOR_PDBQT = WORKDIR / "receptor.pdbqt"
LIGAND_PDB = WORKDIR / "ligand.pdb"
LIGAND_PDBQT = WORKDIR / "ligand.pdbqt"
OUT_PDBQT = WORKDIR / "best_pose.pdbqt"


def clean_rna_sequence(seq: str) -> str:
    seq = seq.strip().upper().replace("T", "U")
    seq = re.sub(r"[^ACGU]", "", seq)
    if len(seq) < 5:
        raise ValueError("Invalid RNA sequence")
    return seq

SEQ = clean_rna_sequence(RNA_SEQUENCE)
FASTA_PATH.write_text(f">{JOB_NAME}\n{SEQ}\n")
print("Sequence:", SEQ, f"({len(SEQ)} nt)")
print("FASTA:", FASTA_PATH)


Sequence: AAAGCGUCGUCCCCACCUC (19 nt)
FASTA: /content/aptamer_docking/aptamer_rf.fasta


## 3. RhoFold+ — предсказание 3D структуры

In [12]:
# === 3. RhoFold+ inference ===
import torch

RHOFOLD_OUT.mkdir(parents=True, exist_ok=True)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

cmd = (
    f"conda run -n rhofold --no-capture-output python {RHOFOLD_DIR}/inference.py "
    f"--input_fas {FASTA_PATH} "
    f"--output_dir {RHOFOLD_OUT} "
    f"--ckpt {CKPT} "
    f"--device {device} "
    f"--relax_steps {RELAX_STEPS} "
)
if USE_SINGLE_SEQ:
    cmd += "--single_seq_pred True "

print("Running RhoFold+...")
!{cmd}

relaxed = sorted(RHOFOLD_OUT.glob("relaxed_*_model.pdb"))
APTAMER_PDB_RAW = relaxed[-1] if relaxed else RHOFOLD_OUT / "unrelaxed_model.pdb"
if not APTAMER_PDB_RAW.exists():
    log = RHOFOLD_OUT / "log.txt"
    raise FileNotFoundError(log.read_text() if log.exists() else "RhoFold+ produced no PDB")

print("Receptor PDB:", APTAMER_PDB_RAW)
ss_ct = RHOFOLD_OUT / "ss.ct"
if ss_ct.exists():
    print("\n--- Predicted 2D (ss.ct, first 400 chars) ---")
    print(ss_ct.read_text()[:400])


Device: cuda:0
Running RhoFold+...
2026-06-11 17:57:26,337 - INFO: Constructing RhoFold
2026-06-11 17:57:27,926 - INFO:     downloading /content/RhoFold/pretrained/rhofold_pretrained_params.pt
Fetching 4 files: 100% 4/4 [00:00<00:00, 11.92it/s]
2026-06-11 17:57:28,415 - INFO:     loading /content/RhoFold/pretrained/rhofold_pretrained_params.pt
2026-06-11 17:57:29,433 - INFO: Input_fas /content/aptamer_docking/aptamer_rf.fasta
2026-06-11 17:57:29,433 - INFO: Input_a3m is None, the modeling will run using single sequence only (input_fas)
2026-06-11 17:57:29,433 - INFO: Started RhoFold Inference
2026-06-11 17:57:29,466 - INFO:     Inference using device cuda:0
2026-06-11 17:57:34,919 - INFO:     Export PDB file to /content/aptamer_docking/rhofold_out/unrelaxed_model.pdb
2026-06-11 17:57:34,919 - INFO: Finished RhoFold Inference in 5.486 seconds
2026-06-11 17:57:34,920 - INFO: Started Amber Relaxation : 200 iterations
2026-06-11 17:57:34,920 - INFO:     AmberRelaxation: Using OpenCL
2026-0

## 4. Подготовка рецептора для Vina

In [13]:
# === 4. Fix / clean / diagnose receptor ===
import numpy as np


def fix_rhofold_pdb(input_pdb, output_pdb):
    lines_out = []
    seen_model = False

    def infer_elem(atom_name):
        n = atom_name.strip()
        if n.startswith("H"): return "H"
        if n.startswith("O"): return "O"
        if n.startswith("P"): return "P"
        if n.startswith("N"): return "N"
        if n.startswith("C"): return "C"
        return n[0] if n else "X"

    with open(input_pdb) as f:
        for line in f:
            if line.startswith("MODEL"):
                if seen_model:
                    break
                seen_model = True
                continue
            if line.startswith("ENDMDL"):
                break
            if not line.startswith(("ATOM", "HETATM")):
                continue
            elem = line[76:78].strip() if len(line) >= 78 else ""
            if not elem:
                atom_name = line[12:16]
                elem = infer_elem(atom_name)
                line = line.rstrip("\n").ljust(78)
                line = line[:76] + f"{elem:>2}" + "\n"
            lines_out.append(line)

    output_pdb.write_text("".join(lines_out) + "END\n")
    return len(lines_out)


def clean_receptor(input_pdb, output_pdb):
    allowed = {"A", "C", "G", "U", "DA", "DC", "DG", "DT", "DU", "RA", "RC", "RG", "RU"}
    water = {"HOH", "WAT", "H2O"}
    kept = []
    with open(input_pdb) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                res = line[17:20].strip()
                if res in water:
                    continue
                if res in allowed:
                    kept.append(line)
    output_pdb.write_text("".join(kept) + "END\n")
    return len(kept)


def diagnose_pdb(pdb_path):
    coords, resnames = [], set()
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                resnames.add(line[17:20].strip())
                coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    c = np.array(coords)
    span = c.max(0) - c.min(0)
    centroid = c.mean(0)
    print("Residues:", sorted(resnames))
    print("Heavy atoms:", len(c))
    print("Span (Å):", span.round(2))
    print("Centroid:", centroid.round(2))
    return c, centroid.tolist()


n1 = fix_rhofold_pdb(APTAMER_PDB_RAW, APTAMER_PDB_FIXED)
n2 = clean_receptor(APTAMER_PDB_FIXED, APTAMER_PDB)
print(f"Fixed: {n1} atoms → Clean: {n2} atoms")

print("\n--- DIAGNOSTIC ---")
_, CENTROID = diagnose_pdb(APTAMER_PDB)
print("\nReceptor for Vina:", APTAMER_PDB)


Fixed: 394 atoms → Clean: 394 atoms

--- DIAGNOSTIC ---
Residues: ['A', 'C', 'G', 'U']
Heavy atoms: 394
Span (Å): [35.45 43.69 28.46]
Centroid: [ 3.44 -4.72 -2.46]

Receptor for Vina: /content/aptamer_docking/aptamer_clean.pdb


## 5. Лиганд: SMILES → PDBQT (RDKit + Meeko, без HTTP)

In [14]:
# === 5. Ligand prep (RDKit + Meeko) ===
import subprocess
import textwrap

script = textwrap.dedent(f"""
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy

smiles = {LIGAND_SMILES!r}
pdb_out = Path({str(LIGAND_PDB)!r})
pdbqt_out = Path({str(LIGAND_PDBQT)!r})

mol = Chem.MolFromSmiles(smiles)
if mol is None:
    raise ValueError(f"RDKit cannot parse SMILES: {{{{smiles}}}}")

mol = Chem.AddHs(mol)
params = AllChem.ETKDGv3()
params.randomSeed = 42
if AllChem.EmbedMolecule(mol, params) != 0:
    if AllChem.EmbedMolecule(mol, AllChem.ETKDG()) != 0:
        raise RuntimeError("3D embedding failed")
AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
Chem.MolToPDBFile(mol, str(pdb_out))

preparator = MoleculePreparation()
setups = preparator.prepare(mol) if hasattr(preparator, "prepare") else preparator(mol)
if not setups:
    raise RuntimeError("Meeko returned no setups")

out = PDBQTWriterLegacy.write_string(setups[0], bad_charge_ok=True)
if isinstance(out, tuple):
    pdbqt_str, is_ok = out[0], out[1]
    if not is_ok:
        err = out[2] if len(out) > 2 else ""
        raise RuntimeError(f"Meeko error: {{{{err}}}}")
else:
    pdbqt_str = out

pdbqt_out.write_text(pdbqt_str)
print("RDKit atoms:", mol.GetNumAtoms())
print("PDB bytes:", pdb_out.stat().st_size)
print("PDBQT bytes:", pdbqt_out.stat().st_size)
""")

proc = subprocess.run([DOCK_PYTHON, "-c", script], capture_output=True, text=True)
print(proc.stdout)
if proc.stderr.strip():
    print("stderr:", proc.stderr[:800])
if proc.returncode != 0:
    raise RuntimeError("Ligand prep failed")

assert LIGAND_PDBQT.stat().st_size > 100, "Empty ligand PDBQT"
print("Ligand prep: OK")


RDKit atoms: 21
PDB bytes: 1967
PDBQT bytes: 1283

Ligand prep: OK


In [15]:
# === 5b. Receptor → PDBQT (OpenBabel) ===
import subprocess

subprocess.run([
    "obabel", str(APTAMER_PDB), "-O", str(RECEPTOR_PDBQT),
    "-xr", "-p", str(LIGAND_PH), "--partialcharge", "gasteiger",
] + (["-h"] if ADD_RECEPTOR_H else []), check=True)

print("receptor.pdbqt:", RECEPTOR_PDBQT.stat().st_size, "bytes")
print("ligand.pdbqt  :", LIGAND_PDBQT.stat().st_size, "bytes")


receptor.pdbqt: 36467 bytes
ligand.pdbqt  : 1283 bytes


## 6. Multi-site AutoDock Vina

In [16]:
# === 6. Utils + multi-site docking ===
import re
import shutil
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path


def read_heavy_coords(pdb_path):
    coords = []
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            elem = line[76:78].strip() if len(line) >= 78 else line[12:16].strip()[0]
            if elem.upper().startswith("H"):
                continue
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.array(coords)


def read_ligand_coords_from_pdbqt(pdbqt_path):
    coords = []
    for line in Path(pdbqt_path).read_text().splitlines():
        if line.startswith(("ATOM", "HETATM")):
            elem = (line[77:79].strip() or line[12:16].strip()[0]).upper()
            if elem.startswith("H"):
                continue
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.array(coords)


def analyze_contacts(receptor_pdb, ligand_pdbqt, cutoff=CONTACT_CUTOFF):
    rec = read_heavy_coords(receptor_pdb)
    lig = read_ligand_coords_from_pdbqt(ligand_pdbqt)
    if len(rec) == 0 or len(lig) == 0:
        return 0, None
    d = np.linalg.norm(rec[:, None, :] - lig[None, :, :], axis=2)
    return int((d <= cutoff).sum()), float(d.min())


def pocket_center(receptor_pdb):
    c = read_heavy_coords(receptor_pdb)
    centroid = c.mean(0)
    dist = np.linalg.norm(c - centroid, axis=1)
    idx = np.argsort(dist)[-max(5, len(c) // 5):]
    return c[idx].mean(0).tolist()


def parse_vina_affinity(stdout_text):
    for line in stdout_text.splitlines():
        m = re.match(r"^\s*1\s+(-?\d+\.\d+)", line)
        if m:
            return float(m.group(1))
    raise ValueError("Cannot parse Vina affinity:\n" + stdout_text[-1500:])


def run_vina_cli(receptor_pdbqt, ligand_pdbqt, center, box, out_pdbqt):
    cx, cy, cz = center
    sx, sy, sz = box
    cmd = [
        str(VINA_BIN),
        "--receptor", str(receptor_pdbqt),
        "--ligand", str(ligand_pdbqt),
        "--center_x", str(cx), "--center_y", str(cy), "--center_z", str(cz),
        "--size_x", str(sx), "--size_y", str(sy), "--size_z", str(sz),
        "--exhaustiveness", str(EXHAUSTIVENESS),
        "--num_modes", str(N_POSES),
        "--out", str(out_pdbqt),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr + "\n" + proc.stdout)
    return parse_vina_affinity(proc.stdout)


sites = {
    "compact": {"center": CENTROID, "box": LOCAL_BOX},
    "pocket": {"center": pocket_center(APTAMER_PDB), "box": LOCAL_BOX},
    "blind": {"center": CENTROID, "box": BLIND_BOX},
}

results = []
for name, site in sites.items():
    tmp_out = WORKDIR / f"docked_{name}.pdbqt"
    aff = run_vina_cli(RECEPTOR_PDBQT, LIGAND_PDBQT, site["center"], site["box"], tmp_out)
    contacts, min_dist = analyze_contacts(APTAMER_PDB, tmp_out)
    results.append({
        "site": name,
        "center": site["center"],
        "box": site["box"],
        "affinity": aff,
        "contacts": contacts,
        "min_dist": min_dist,
        "pose": tmp_out,
    })
    print(f"{name:7s} | affinity={aff:7.3f} | contacts={contacts:3d} | min_dist={min_dist:.2f} Å")

centers = np.array([r["center"] for r in results])
consensus = centers.mean(0)
for r in results:
    r["dist_to_consensus"] = float(np.linalg.norm(np.array(r["center"]) - consensus))

filtered = [r for r in results if r["dist_to_consensus"] <= CONSENSUS_CUTOFF] or results
filtered.sort(key=lambda x: (x["affinity"], -x["contacts"]))
best = filtered[0]
shutil.copy2(best["pose"], OUT_PDBQT)

print("\n=== BEST ===")
print("Site:", best["site"])
print("Affinity (kcal/mol):", round(best["affinity"], 3))
print("Contacts:", best["contacts"])
print("Min dist (Å):", round(best["min_dist"], 2))
print("Pose:", OUT_PDBQT)

if best["contacts"] < MIN_CONTACTS:
    print(f"WARNING: contacts < {MIN_CONTACTS} — попробуйте unrelaxed PDB или box 26–28 Å")

pd.DataFrame(results).to_csv(WORKDIR / "multisite_results.csv", index=False)
print("Saved:", WORKDIR / "multisite_results.csv")


compact | affinity= -4.689 | contacts=160 | min_dist=2.80 Å
pocket  | affinity= -4.559 | contacts=145 | min_dist=2.88 Å
blind   | affinity= -4.634 | contacts=142 | min_dist=2.83 Å

=== BEST ===
Site: compact
Affinity (kcal/mol): -4.689
Contacts: 160
Min dist (Å): 2.8
Pose: /content/aptamer_docking/best_pose.pdbqt
Saved: /content/aptamer_docking/multisite_results.csv


## 7. 3D-визуализация

In [18]:
# === 7. Visualization (detailed) ===
import numpy as np
import py3Dmol
from pathlib import Path

CONTACT_CUTOFF_VIZ = 4.0  # Å

def _heavy_coords_pdb(path):
    coords, resids = [], []
    with open(path) as f:
        for line in f:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            elem = line[76:78].strip() if len(line) >= 78 else line[12:16].strip()[0]
            if elem.upper().startswith("H"):
                continue
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            resids.append(int(line[22:26]))
    return np.array(coords), resids

def _heavy_coords_pdbqt(path):
    coords = []
    for line in Path(path).read_text().splitlines():
        if line.startswith(("ATOM", "HETATM")):
            elem = (line[77:79].strip() or line[12:16].strip()[0]).upper()
            if elem.startswith("H"):
                continue
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.array(coords)

def contact_residue_ids(receptor_pdb, ligand_pdbqt, cutoff=CONTACT_CUTOFF_VIZ):
    rec, resids = _heavy_coords_pdb(receptor_pdb)
    lig = _heavy_coords_pdbqt(ligand_pdbqt)
    if len(rec) == 0 or len(lig) == 0:
        return [], lig.mean(0) if len(lig) else np.zeros(3)
    d = np.linalg.norm(rec[:, None, :] - lig[None, :, :], axis=2)
    mask = (d <= cutoff).any(axis=1)
    contact_ids = sorted(set(r for r, m in zip(resids, mask) if m))
    return contact_ids, lig.mean(0)

rec_pdb = str(APTAMER_PDB)
pose_pdbqt = str(OUT_PDBQT)
rec_text = open(rec_pdb).read()
pose_text = open(pose_pdbqt).read()

contact_ids, lig_center = contact_residue_ids(rec_pdb, pose_pdbqt)

try:
    aff = best["affinity"]
    site_name = best["site"]
    n_contacts = best["contacts"]
    min_dist = best["min_dist"]
except NameError:
    aff, site_name, n_contacts, min_dist = None, "compact", len(contact_ids), None

title = "Aptamer–ligand docking"
if aff is not None:
    title += f" | {site_name} | {aff:.2f} kcal/mol | {n_contacts} contacts"
print(title)
print("Contact residues (≤4 Å):", contact_ids)

view = py3Dmol.view(width=960, height=620, viewergrid=(1, 2))

# --- Panel 1: overview ---
view.addModel(rec_text, "pdb", viewer=(0, 0))
view.setStyle({"model": 0}, {"cartoon": {"color": "#cfd8dc", "opacity": 0.55}}, viewer=(0, 0))

base_colors = {"A": "#ef5350", "U": "#42a5f5", "G": "#66bb6a", "C": "#ffa726"}
for resn, color in base_colors.items():
    view.setStyle({"model": 0, "resn": resn}, {"cartoon": {"color": color, "opacity": 0.35}}, viewer=(0, 0))

if contact_ids:
    view.setStyle(
        {"model": 0, "resi": contact_ids},
        {"stick": {"colorscheme": "orangeCarbon", "radius": 0.18},
         "cartoon": {"color": "#ff7043", "opacity": 0.95}},
        viewer=(0, 0),
    )

view.addModel(pose_text, "pdb", viewer=(0, 0))
view.setStyle(
    {"model": 1},
    {"stick": {"colorscheme": "magentaCarbon", "radius": 0.22},
     "sphere": {"scale": 0.28, "colorscheme": "magentaCarbon"}},
    viewer=(0, 0),
)

view.addSphere(
    {"center": {"x": float(lig_center[0]), "y": float(lig_center[1]), "z": float(lig_center[2])},
     "radius": 1.6, "color": "yellow", "alpha": 0.25},
    viewer=(0, 0),
)

view.setBackgroundColor("#1a1a2e", viewer=(0, 0))
view.addLabel(
    "Overview",
    {"position": {"x": 0, "y": 0, "z": 0}, "backgroundColor": "#16213e",
     "fontColor": "white", "fontSize": 14, "showBackground": True},
    viewer=(0, 0),
)
view.zoomTo({"model": 1}, viewer=(0, 0))

# --- Panel 2: binding site close-up ---
view.addModel(rec_text, "pdb", viewer=(0, 1))
view.setStyle({"model": 0}, {"line": {"color": "#90a4ae", "opacity": 0.3}}, viewer=(0, 1))

if contact_ids:
    view.setStyle(
        {"model": 0, "resi": contact_ids},
        {"stick": {"colorscheme": "whiteCarbon", "radius": 0.2}},
        viewer=(0, 1),
    )
    view.addSurface(
        py3Dmol.VDW,
        {"opacity": 0.55, "color": "#26c6da"},
        {"model": 0, "resi": contact_ids},
        viewer=(0, 1),
    )

view.addModel(pose_text, "pdb", viewer=(0, 1))
view.setStyle(
    {"model": 1},
    {"stick": {"colorscheme": "greenCarbon", "radius": 0.25},
     "sphere": {"scale": 0.32, "colorscheme": "greenCarbon"}},
    viewer=(0, 1),
)

view.setBackgroundColor("#0f0f1a", viewer=(0, 1))
label_txt = "Binding site"
if aff is not None:
    label_txt += f"\nΔG = {aff:.2f} kcal/mol"
if min_dist is not None:
    label_txt += f"\nmin dist = {min_dist:.2f} Å"
view.addLabel(
    label_txt,
    {"position": {"x": float(lig_center[0]), "y": float(lig_center[1]), "z": float(lig_center[2])},
     "backgroundColor": "#1b4332", "fontColor": "#d8f3dc",
     "fontSize": 13, "showBackground": True, "alignment": "center"},
    viewer=(0, 1),
)
view.zoomTo({"model": 1}, viewer=(0, 1))

view.show()

# --- Save PNG (Colab) ---
png_path = WORKDIR / "docking_visualization.png"
try:
    png_data = view.get_png() if hasattr(view, "get_png") else view.png()
    png_path.write_bytes(png_data)
    print("Saved:", png_path)
except Exception as e:
    print("PNG save skipped:", e)
    try:
        png_data = view._repr_png_()
        if png_data:
            png_path.write_bytes(png_data)
            print("Saved (fallback):", png_path)
    except Exception as e2:
        print("PNG fallback failed:", e2)

Aptamer–ligand docking | compact | -4.69 kcal/mol | 160 contacts
Contact residues (≤4 Å): [6, 7, 8, 9]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

PNG save skipped: memoryview: a bytes-like object is required, not 'view'
PNG fallback failed: <class 'py3Dmol.view'> object has no attribute '_repr_png_'
